# GenAI Pipeline — Batch Output

Retrieves results from an Anthropic Message Batch submitted by `genai_scope_batchinput.ipynb`,
parses the structured outputs, compares against ground-truth labels, and saves results to Excel.

### 1. Imports and Configuration

In [ ]:
import duckdb
import pandas as pd
import json
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

DB_PATH = "../publications.db"
BATCH_DIR = Path("batch_jobs")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

### 2. Load Batch Metadata

Set `BATCH_ID` to a specific batch ID string (e.g. `"msgbatch_01AbCdEf..."`),
or leave as `None` to pick up the most recently created batch automatically.

In [ ]:
BATCH_ID = None  # ← set to a specific batch ID, or None to use the most recent

if BATCH_ID is None:
    batch_files = sorted(BATCH_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime)
    if not batch_files:
        raise FileNotFoundError("No batch job files in batch_jobs/. Run genai_scope_batchinput.ipynb first.")
    metadata_path = batch_files[-1]
    print(f"Using most recent batch file: {metadata_path.name}")
else:
    metadata_path = BATCH_DIR / f"{BATCH_ID}.json"

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
batch_id = metadata["batch_id"]
INCLUDE_REASONING = metadata["include_reasoning"]

print(f"Batch ID:   {batch_id}")
print(f"Model:      {metadata['model']}")
print(f"Prompt:     {metadata['prompt_path']}")
print(f"Records:    {metadata['n_records']}")
print(f"Created:    {metadata.get('created_at', 'unknown')}")
print(f"Reasoning:  {INCLUDE_REASONING}")

### 3. Check Batch Status

Re-run this cell until `processing_status` is `ended`. The batch typically completes
well within 24 hours. Do not proceed to section 4 until it is complete.

In [ ]:
batch = client.messages.batches.retrieve(batch_id)

print(f"Processing status: {batch.processing_status}")
print(f"Request counts:    {batch.request_counts}")

if batch.processing_status != "ended":
    print("\nBatch is not yet complete — re-run this cell to check again.")
else:
    print("\nBatch complete — proceed to section 4 to retrieve results.")

### 4. Retrieve and Parse Results

In [ ]:
if batch.processing_status != "ended":
    raise RuntimeError("Batch is not yet complete. Check status in section 3.")

raw_results = []
for result in client.messages.batches.results(batch_id):
    if result.result.type == "succeeded":
        content = result.result.message.content
        tool_block = next((b for b in content if b.type == "tool_use"), None)
        if tool_block:
            record = dict(tool_block.input)
            record["id"] = result.custom_id
            record["status"] = "ok"
        else:
            record = {"id": result.custom_id, "status": "parse_error", "error": "no tool_use block"}
    else:
        record = {
            "id": result.custom_id,
            "status": result.result.type,
            "error": str(getattr(result.result, "error", "")),
        }
    raw_results.append(record)

results_df = pd.DataFrame(raw_results)

# Rename LLM output columns to add _LLM suffix (matches genai_testing.ipynb convention)
non_llm_cols = {"id", "status", "error"}
results_df = results_df.rename(columns={c: f"{c}_LLM" for c in results_df.columns if c not in non_llm_cols})

print(f"Total results:  {len(results_df)}")
print(f"Successful:     {(results_df['status'] == 'ok').sum()}")
print(f"Errors:         {(results_df['status'] != 'ok').sum()}")
results_df.head()

### 5. Load Ground Truth and Compare

Merges LLM predictions against the labelled records in `publications_raw`.
Records without a ground-truth label (scope is NA) are excluded from accuracy metrics.

In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)
df_truth = con.sql("SELECT id, title, abstract, scope, pillar FROM publications_raw").df()
con.close()

# Filter to only the records that were in this batch
batch_ids = set(metadata["dataset_ids"])
df_truth = df_truth[df_truth["id"].isin(batch_ids)].reset_index(drop=True)

result_cols = ["id", "status", "scope_LLM", "confidence_LLM",
               "plant_based_LLM", "fermentation_LLM", "cultivated_LLM", "cross_cutting_LLM"]
if INCLUDE_REASONING and "reasoning_LLM" in results_df.columns:
    result_cols.insert(result_cols.index("confidence_LLM") + 1, "reasoning_LLM")

comparison = df_truth.merge(results_df[[c for c in result_cols if c in results_df.columns or c == "id"]], on="id", how="left")
comparison["correct_scope"] = comparison["scope"] == comparison["scope_LLM"]

# Derive pillar_LLM from boolean flags (same logic as genai_testing.ipynb):
# CC if multiple pillar flags are True, or if only cross_cutting_LLM is True
pillar_flags = ["plant_based_LLM", "fermentation_LLM", "cultivated_LLM"]
comparison["pillar_LLM"] = comparison[pillar_flags + ["cross_cutting_LLM"]].apply(
    lambda r: "CC" if r[pillar_flags].sum() > 1 or (r["cross_cutting_LLM"] and r[pillar_flags].sum() == 0)
              else "PB" if r["plant_based_LLM"]
              else "F"  if r["fermentation_LLM"]
              else "CM" if r["cultivated_LLM"]
              else "NA",
    axis=1
)
comparison["correct_pillar"] = comparison["pillar"] == comparison["pillar_LLM"]

comparison.head()

In [ ]:
has_truth = comparison["scope"].notna()

print(f"Scope accuracy:  {comparison.loc[has_truth, 'correct_scope'].mean():.0%}  (n={has_truth.sum()})")
print(f"Pillar accuracy: {comparison.loc[has_truth, 'correct_pillar'].mean():.0%}  (n={has_truth.sum()})")

both_in = (comparison["scope"] == "in") & (comparison["scope_LLM"] == "in")
print(f"Pillar accuracy — both in-scope: {comparison.loc[both_in, 'correct_pillar'].mean():.0%}  (n={both_in.sum()})")

display_cols = ["id", "title", "abstract", "scope", "scope_LLM", "confidence_LLM"]
if INCLUDE_REASONING and "reasoning_LLM" in comparison.columns:
    display_cols.append("reasoning_LLM")
display_cols += ["correct_scope", "pillar", "pillar_LLM", "correct_pillar",
                 "plant_based_LLM", "fermentation_LLM", "cultivated_LLM", "cross_cutting_LLM"]

comparison[display_cols]

### 6. Save to Excel

In [ ]:
save_dir = Path("batch_results") / batch_id  # ← change if you prefer a different output location
save_dir.mkdir(parents=True, exist_ok=True)

model_name = metadata["model"]
metrics_df = pd.DataFrame([
    {"metric": "scope_accuracy",                "value": f"{comparison.loc[has_truth, 'correct_scope'].mean():.0%}",               "n": int(has_truth.sum())},
    {"metric": "pillar_accuracy",               "value": f"{comparison.loc[has_truth, 'correct_pillar'].mean():.0%}",              "n": int(has_truth.sum())},
    {"metric": "pillar_accuracy_both_in_scope", "value": f"{comparison.loc[both_in, 'correct_pillar'].mean():.0%}", "n": int(both_in.sum())},
])

output_path = save_dir / f"{model_name}_results.xlsx"
with pd.ExcelWriter(output_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="comparison", index=False)
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)

print(f"Saved to {output_path}")